# CareFlow — Week 4 Final Analysis
This notebook analyzes patient paths, activity wait times, loop-backs, and the slowest activity transitions.

In [1]:
import pandas as pd

df = pd.read_csv("careflow_patients_data.csv")
print("Rows and columns:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head())

Rows and columns: (1134, 3)
Columns: ['Case ID', 'Activity Name', 'Timestamp']


,Case ID,Activity Name,Timestamp
0,P001,Registration,08:20
1,P001,Vitals,08:53
2,P001,Triage,09:25
3,P001,Doctor,09:49
4,P001,Billing Counter,10:06


In [2]:
print("Missing values:")
print(df[["Case ID", "Activity Name", "Timestamp"]].isna().sum())

df["Timestamp"] = pd.to_datetime(df["Timestamp"], errors="coerce")

print("\nTimestamps that could not be parsed:", df["Timestamp"].isna().sum())
print("Timestamp range:", df["Timestamp"].min(), "to", df["Timestamp"].max())

Missing values:
Case ID          0
Activity Name    0
Timestamp        0
dtype: int64


C:\Users\91822\AppData\Local\Temp\ipykernel_15880\3425905270.py:4: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Timestamp"] = pd.to_datetime(df["Timestamp"], errors="coerce")



Timestamps that could not be parsed: 0
Timestamp range: 2026-11-07 08:00:00 to 2026-11-07 19:04:00


## Day 2 — Top Patient Paths

Sort patient activities by timestamp, build an ordered path for each case, and identify the five most common paths.

In [3]:
import pandas as pd

# Load the dataset
df = pd.read_csv("careflow_patients_data.csv")

# Prepare and sort timestamps
df["Timestamp"] = pd.to_datetime(df["Timestamp"])
df = df.sort_values(["Case ID", "Timestamp"])

# Build and count each patient's exact activity sequence
patient_paths = df.groupby("Case ID")["Activity Name"].agg(tuple)
top_5_paths = patient_paths.value_counts().head(5)

# Display the five most common paths
top_5_results = pd.DataFrame({
    "Patient Count": top_5_paths.values,
    "Path": [" → ".join(path) for path in top_5_paths.index]
})

top_5_results.index = range(1, len(top_5_results) + 1)
top_5_results.index.name = "Rank"

display(top_5_results)

C:\Users\91822\AppData\Local\Temp\ipykernel_4956\664647128.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Timestamp"] = pd.to_datetime(df["Timestamp"])


,Patient Count,Path
Rank,,
1,50,Registration → Vitals → Triage → Doctor → Bill...
2,15,Registration → Vitals → Triage → Doctor → Bill...
3,15,Registration → Vitals → Triage → Doctor → Bill...
4,13,Registration → Vitals → Triage → Doctor → Bill...
5,4,Registration → Vitals → Triage → Doctor → Bill...


## Day 3 — Wait Times Between Activities

Calculate the elapsed time from each activity to the next for each patient, then summarize average waits by activity transition.

In [2]:
import pandas as pd

# Load and prepare the data
df = pd.read_csv("careflow_patients_data.csv")
df["Timestamp"] = pd.to_datetime(df["Timestamp"])
df = df.sort_values(["Case ID", "Timestamp"]).copy()

# Get the previous activity and timestamp for each patient
df["Previous Activity"] = df.groupby("Case ID")["Activity Name"].shift(1)
df["Previous Timestamp"] = df.groupby("Case ID")["Timestamp"].shift(1)

# Calculate time between consecutive activities, in hours
df["Wait Time (Hours)"] = (
    (df["Timestamp"] - df["Previous Timestamp"]).dt.total_seconds() / 3600
)

# Exclude each patient's first activity, which has no preceding activity
waits = df.dropna(subset=["Previous Timestamp"]).copy()

# Summarize waits for each activity transition
wait_summary = (
    waits.groupby(["Previous Activity", "Activity Name"])["Wait Time (Hours)"]
         .agg(["mean", "median", "count"])
         .sort_values("mean", ascending=False)
         .reset_index()
)

display(wait_summary.head(10))
print(f"Overall average wait: {waits['Wait Time (Hours)'].mean():.2f} hours")

C:\Users\91822\AppData\Local\Temp\ipykernel_2272\2849061708.py:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Timestamp"] = pd.to_datetime(df["Timestamp"])


,Previous Activity,Activity Name,mean,median,count
0,Billing Counter,Doctor,1.341667,1.366667,4
1,X-Ray report,Pathology report,1.049735,1.116667,63
2,X-Ray report,Doctor,1.033333,1.183333,15
3,Triage,Doctor,1.012500,1.041667,116
4,X-Ray,X-Ray report,0.990385,1.008333,78
5,Pathology report,Doctor,0.952821,0.966667,65
6,Pathology,Pathology report,0.878704,0.791667,18
7,Vitals,Triage,0.340833,0.350000,100
8,Billing Counter,Pathology,0.338066,0.350000,81
9,Doctor,Billing Counter,0.331833,0.316667,100


Overall average wait: 0.55 hours
